In [1]:
import dspy
from datasets import load_dataset

# 1. Initialisierung der Modelle (lokal betrieben)
# Annahme: Ein Embedding-Modell-Server läuft auf Port 8081
embedder = dspy.Embedder(
    "openai/qwen3-embedding:0.6b", 
    api_base="http://localhost:11434/v1", 
    api_key="no_key_needed", 
    batch_size=100
)

local_llm = dspy.LM(
    "openai/unsloth/gemma-3-4b-it-GGUF:Q4_K_M", 
    api_base="http://localhost:8080/v1", 
    api_key="no_key_needed",
    temperature=0.1,
    cache=False,
)

dspy.configure(lm=local_llm, embedder=embedder)

In [2]:
# Laden des Datensatzes
dataset = load_dataset("embedding-data/simple-wiki", split="train[:1000]")
documents = [" ".join(doc['set']) for doc in dataset] 

len (documents)

1000

In [3]:
# Aus den Dokumenten Chunks erzeugen
chunk_size = 1024
all_chunks = []

for doc in documents:
    # Anzahl der Chunks für dieses Dokument
    for i in range(0, len(doc), chunk_size):
        chunk = doc[i:i+chunk_size]
        if len(chunk) > 100:        # nur Chunks > 100 Zeichen übernehmen
            all_chunks.append(chunk)

In [4]:
# Erstellen der Qdrant Collection (nur falls sie nicht existiert)
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

# Qdrant Client erstellen
client = QdrantClient(host="localhost", port=6333)

collection_name = "simple_wiki_rag"
embedding_dim = 1024  # Dimension für embedding-gemma-Modell

# Prüfen, ob die Collection existiert
if not client.collection_exists(collection_name):
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(
            size=embedding_dim,
            distance=Distance.COSINE
        )
    )
    print(f"Collection '{collection_name}' created successfully.")
else:
    print(f"Collection '{collection_name}' already exists.")

Collection 'simple_wiki_rag' created successfully.


In [6]:
# Embedden und Indexieren der Dokumente
embeddings = embedder(all_chunks)
len(embeddings)

974

In [7]:
from qdrant_client.models import PointStruct

points = [
    PointStruct(id=i, vector=vec, payload={"text": chunk})
    for i, (chunk, vec) in enumerate(zip(all_chunks, embeddings))
]

In [8]:
# Punkte in Qdrant hochladen (in Batches für große Daten)
batch_size = 100
for i in range(0, len(points), batch_size):
    batch = points[i:i+batch_size]
    client.upsert(
        collection_name=collection_name,
        points=batch
    )

print(f"{len(points)} Chunks erfolgreich in Qdrant gespeichert.")

974 Chunks erfolgreich in Qdrant gespeichert.


In [9]:
# 1. Definition des Qdrant Retrievers
class QdrantRetriever(dspy.Retrieve):
    def __init__(self, client, collection_name, embedder, k=3):
        self._client = client
        self._collection_name = collection_name
        self._embedder = embedder
        self._k = k
        super().__init__()

    def forward(self, query_or_queries, k=None):
        k = k if k is not None else self._k
        # Embedden der Suchanfrage
        query_embeddings = self._embedder(query_or_queries)

        # Suche in Qdrant
        results = [
            self._client.query_points(
                collection_name=collection_name,
                query=query_embeddings,
                limit=3,
            ) for emb in query_embeddings
        ]

        passages = [dspy.Prediction(long_text=p.payload["text"]) for p in results[0].points]

        return passages

In [10]:
rm = QdrantRetriever(client, collection_name, embedder)

dspy.settings.configure(rm=rm)

In [11]:
from dspy.evaluate import Evaluate
from dspy.datasets.hotpotqa import HotPotQA

# Definition der RAG-Klasse (basierend auf vorherigen Tagen)
class RAG(dspy.Module):
    def __init__(self, num_passages=3):
        super().__init__()
        self.retrieve = dspy.Retrieve(k=num_passages)
        self.generate_answer = dspy.ChainOfThought("context, question -> answer")

    def forward(self, question):
        context = self.retrieve(question).passages
        prediction = self.generate_answer(context=context, question=question)
        return dspy.Prediction(context=context, answer=prediction.answer)

# Laden eines Beispieldatensatzes (z.B. HotPotQA)
dataset = HotPotQA(train_seed=1, train_size=20, eval_seed=42, dev_size=20, test_size=0)
trainset = [x.with_inputs('question') for x in dataset.train]
devset = [x.with_inputs('question') for x in dataset.dev]

In [12]:
from dspy.teleprompt import BootstrapFewShot
from dspy.evaluate import answer_exact_match

# Definition einer zusammengesetzten Metrik
# Prüft, ob die Antwort korrekt ist UND ob der Kontext relevant ist (optional)
def validate_context_and_answer(example, pred, trace=None):
    answer_match = dspy.evaluate.answer_exact_match(example, pred)
    return answer_match

# Initialisierung des Optimizers
teleprompter = BootstrapFewShot(
    metric=validate_context_and_answer,
    max_bootstrapped_demos=4,  # Anzahl der zu generierenden Beispiele pro Prompt
    max_labeled_demos=4       # Anzahl der manuellen Beispiele (falls vorhanden)
)

In [13]:
# Kompilierung des RAG-Programms
# Der Prozess durchläuft das Trainset und optimiert die Prompts
print("Starte Kompilierung...")
compiled_rag = teleprompter.compile(student=RAG(), trainset=trainset)
print("Kompilierung abgeschlossen.")

Starte Kompilierung...


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [02:41<00:00,  8.05s/it]

Bootstrapped 0 full traces after 19 examples for up to 1 rounds, amounting to 20 attempts.
Kompilierung abgeschlossen.


In [14]:
# Definition der Evaluierungsfunktion
evaluator = Evaluate(devset=devset, metric=answer_exact_match, num_threads=1, display_progress=True, display_table=0)

# Testen des unkompilierten Programms (Zero-Shot)
print("Evaluation: Unkompiliertes RAG")
uncompiled_rag = RAG()
evaluator(uncompiled_rag)

# Testen des kompilierten Programms (Few-Shot Optimized)
print("Evaluation: Kompiliertes RAG")
evaluator(compiled_rag)

Evaluation: Unkompiliertes RAG
Average Metric: 0.00 / 20 (0.0%): 100%|███████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [03:02<00:00,  9.12s/it]

2025/12/07 14:24:08 INFO dspy.evaluate.evaluate: Average Metric: 0 / 20 (0.0%)



Evaluation: Kompiliertes RAG
Average Metric: 3.00 / 20 (15.0%): 100%|██████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [02:36<00:00,  7.85s/it]

2025/12/07 14:26:45 INFO dspy.evaluate.evaluate: Average Metric: 3 / 20 (15.0%)


EvaluationResult(score=15.0, results=<list of 20 results>)

In [15]:
# Hier muss Retrieve.dump_state etwas geändert werden da sonst ein Fehler auftritt, den ich 
# anders nicht gelöst bekommen habe ....
original_dump_state = dspy.Retrieve.dump_state
def patched_dump_state(self, json_mode=True):
    # Call the original method without arguments
    return original_dump_state(self)
dspy.Retrieve.dump_state = patched_dump_state

# Speichern des kompilierten Zustands
compiled_rag.save("compiled_rag_v1.json")

In [16]:
# ... Simulation eines Neustarts der Anwendung / Produktionsumgebung ...

# Laden des Zustands in eine frische Instanz
production_rag = RAG()
production_rag.load("compiled_rag_v1.json")

In [17]:
# Definition der Evaluierungsfunktion
evaluator = Evaluate(devset=devset, metric=answer_exact_match, num_threads=1, display_progress=True, display_table=0)

# Testen des unkompilierten Programms (Zero-Shot)
print("Evaluation: Unkompiliertes RAG")
uncompiled_rag = RAG()
evaluator(uncompiled_rag)

# Testen des kompilierten Programms (Few-Shot Optimized)
print("Evaluation: Kompiliertes RAG")
evaluator(production_rag)

Evaluation: Unkompiliertes RAG
Average Metric: 0.00 / 20 (0.0%): 100%|███████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [03:05<00:00,  9.29s/it]

2025/12/07 14:31:18 INFO dspy.evaluate.evaluate: Average Metric: 0 / 20 (0.0%)



Evaluation: Kompiliertes RAG
Average Metric: 3.00 / 20 (15.0%): 100%|██████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [02:31<00:00,  7.57s/it]

2025/12/07 14:33:49 INFO dspy.evaluate.evaluate: Average Metric: 3 / 20 (15.0%)


EvaluationResult(score=15.0, results=<list of 20 results>)